# Loan Balance Cleaning
This notebook processes `Raw_Loan_Balance.csv` to create a cleansed layer for Power BI.
- **Staging**: Inspect data, analyze missing values and distributions, standardize formats, validate key columns, log issues.
- **Cleansed**: Convert numeric columns, create issue flags, filter CANCELLED loans, resolve duplicates, save to `cleansed/cleansed_loan_balances.csv`.


## Staging: Imports and Data Loading
Load raw data and required libraries.

In [1]:
import pandas as pd

# Load raw data with semicolon delimiter
df = pd.read_csv('Raw_Loan_Balance.csv', delimiter=';')


## Staging: Data Inspection
Inspect data types, missing values, unique values, and distributions.

In [2]:
# Display first 5 rows
print("First 5 rows of raw data:")
print(df.head())

# Display data types and non-null counts
print("\nData info:")
print(df.info())

# Display missing values per column
print("\nNumber of NaN values per column:")
print(df.isna().sum())

First 5 rows of raw data:
   LoanAccountBalanceId  SourceId  BalanceDateId  LoanAccountId  ProductId  \
0                  9937         8       20230604              2        293   
1                 92431         8       20230605              2        293   
2                174824         8       20230606              2        293   
3                257217         8       20230607              2        293   
4                339610         8       20230608              2        293   

   AccountCurrencyId  AccountStatusId  NumOfTransactions  \
0                 49              331                  0   
1                 49              331                  0   
2                 49              331                  0   
3                 49              331                  0   
4                 49              331                  0   

   NetTransactionAmount  NetTransactionAmountSek AccruedInterest  \
0                     0                        0               0   
1       

In [3]:
# Analyze unique values and frequency distribution for each column
for col in df.columns:
    print(f"\nAnalyzing {col}")
    print("Unique values:", df[col].unique()[:10])  # First 10 unique values
    print("Most frequent values:", df[col].value_counts().head(10))  # Top 10 most frequent
    print("-" * 50)



Analyzing LoanAccountBalanceId
Unique values: [  9937  92431 174824 257217 339610 422003 504396 586789 669182 751574]
Most frequent values: LoanAccountBalanceId
1893935    1
619423     1
537030     1
454637     1
372244     1
289851     1
207458     1
125065     1
42614      1
784163     1
Name: count, dtype: int64
--------------------------------------------------

Analyzing SourceId
Unique values: [8]
Most frequent values: SourceId
8    4657
Name: count, dtype: int64
--------------------------------------------------

Analyzing BalanceDateId
Unique values: [20230604 20230605 20230606 20230607 20230608 20230615 20230616 20230617
 20230618 20230619]
Most frequent values: BalanceDateId
20230604    346
20230605    346
20230606    346
20230607    346
20230608    346
20230615    346
20230616    346
20230617    346
20230618    346
20230619    346
Name: count, dtype: int64
--------------------------------------------------

Analyzing LoanAccountId
Unique values: [ 2  5  6  7  8  9 10 11 12 

## Removing Columns with Only Zero Values

Certain columns contain only zero values across all rows, making them redundant for analysis. These columns are removed to improve data efficiency and eliminate unnecessary variables.


In [4]:
# Identify columns that contain only zeros
zero_columns = [col for col in df.columns if df[col].nunique() == 1 and df[col].unique()[0] == 0]

# Drop them from the dataset
df.drop(columns=zero_columns, inplace=True)

# Verify remaining columns
print("Columns retained:", df.columns)


Columns retained: Index(['LoanAccountBalanceId', 'SourceId', 'BalanceDateId', 'LoanAccountId',
       'ProductId', 'AccountCurrencyId', 'AccountStatusId', 'AccruedInterest',
       'AccruedInterestSEK', 'Balance', 'BalanceSek', 'PrecedingId'],
      dtype='object')


In [5]:
print(df.isna().sum())

LoanAccountBalanceId      0
SourceId                  0
BalanceDateId             0
LoanAccountId             0
ProductId                 0
AccountCurrencyId         0
AccountStatusId           0
AccruedInterest           0
AccruedInterestSEK        0
Balance                   0
BalanceSek                0
PrecedingId             346
dtype: int64


## Staging: Standardize Formats
Convert date and numeric columns to consistent formats.

In [6]:
# Convert BalanceDateId to datetime
df['BalanceDateId'] = pd.to_datetime(df['BalanceDateId'], format='%Y%m%d', errors='coerce')

# Convert numeric columns with commas to float
numeric_columns = ['Balance', 'AccruedInterest', 'AccruedInterestSEK', 'BalanceSek']
for col in numeric_columns:
    try:
        df[col] = df[col].astype(str).str.replace(',', '.').astype(float)
    except Exception as e:
        print(f"Error converting {col}: {e}")

# Handle NaN in PrecedingId and convert to int
df['PrecedingId'] = df['PrecedingId'].fillna(0).astype(int)

# Verify updated data types
print("\nUpdated data types:")
print(df.dtypes)


Updated data types:
LoanAccountBalanceId             int64
SourceId                         int64
BalanceDateId           datetime64[ns]
LoanAccountId                    int64
ProductId                        int64
AccountCurrencyId                int64
AccountStatusId                  int64
AccruedInterest                float64
AccruedInterestSEK             float64
Balance                        float64
BalanceSek                     float64
PrecedingId                      int64
dtype: object


# Cleansed Layer
## Final Review and Saving as CSV

Since no additional data cleansing is required for this file, a final validation step is performed to confirm its integrity. The dataset is then saved as a CSV file for further use in analysis or reporting.


In [7]:
# Perform final review
print(df.info())  # Verify data types and structure
print(df.head())  # Quick preview of the first rows




<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4657 entries, 0 to 4656
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   LoanAccountBalanceId  4657 non-null   int64         
 1   SourceId              4657 non-null   int64         
 2   BalanceDateId         4657 non-null   datetime64[ns]
 3   LoanAccountId         4657 non-null   int64         
 4   ProductId             4657 non-null   int64         
 5   AccountCurrencyId     4657 non-null   int64         
 6   AccountStatusId       4657 non-null   int64         
 7   AccruedInterest       4657 non-null   float64       
 8   AccruedInterestSEK    4657 non-null   float64       
 9   Balance               4657 non-null   float64       
 10  BalanceSek            4657 non-null   float64       
 11  PrecedingId           4657 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(7)
memory usage: 436.7 KB
None
   Lo

In [8]:
# Save the cleaned dataset as a CSV file
df.to_csv("cleansed_loan_balance.csv", index=False)

In [9]:
print(df.columns)

Index(['LoanAccountBalanceId', 'SourceId', 'BalanceDateId', 'LoanAccountId',
       'ProductId', 'AccountCurrencyId', 'AccountStatusId', 'AccruedInterest',
       'AccruedInterestSEK', 'Balance', 'BalanceSek', 'PrecedingId'],
      dtype='object')
